# search-compare — Exemplo de uso

Este notebook demonstra como usar a biblioteca `search_compare` para:

1. Construir duas strings de busca PICOC
2. Executar as buscas na Scopus
3. Comparar os resultados e identificar documentos em comum, exclusivos de cada string

## Pré-requisitos

Defina a variável de ambiente `SCOPUS_API_KEY` antes de executar.

Ou crie um arquivo `.env` na raiz do projeto com sua chave da API Scopus:

```
SCOPUS_API_KEY=sua_chave_aqui
```


## 1. Imports

In [32]:
import sys
sys.path.insert(0, "../src")  # necessário apenas quando executado fora do ambiente instalado

from search_compare import (
    QueryBuilder,
    ScopusClient,
    ScopusQueryBuilder,
    compare,
)

## 2. Configurar os filtros Scopus

O `ScopusQueryBuilder` é compartilhado entre os dois builders e mantém os filtros configurados.

Opções disponíveis em `configure`:
- `field`: `"TITLE-ABS-KEY"` (padrão), `"TITLE"`, `"ABS"`, `"KEY"`
- `year_min` / `year_max`: intervalo de publicação (exclusivo)
- `language`: ex. `"english"`
- `doc_type`: `"ar"` (artigo), `"re"` (review), `"cp"` (conference paper), etc.

In [33]:
scopus_qb = ScopusQueryBuilder()
scopus_qb.configure(
    field="TITLE-ABS-KEY",
    year_min=2020,
    year_max=2026,
    language="english",
    doc_type=["ar", "cp"]
)

## 3. Construir as strings de busca PICOC

Cada componente PICOC aceita múltiplos termos, que são combinados com `OR`.
Os componentes definidos são combinados com `AND`.

Termos com espaços devem estar entre aspas: `'"embedded system"'`

In [34]:
# b1 = QueryBuilder(backend=scopus_qb)
# b1.population('"embedded software"', '"embedded system"', 'microcontroller')
# b1.intervention('architecture', '"design pattern"')
# b1.outcome('maintainability', 'maintainable', 'testability', 'testable')

b1 = QueryBuilder(backend=scopus_qb)
b1.population('MCU', 'microcontroller', '"embedded software"', '"embedded system"')
b1.intervention('architecture', '"design pattern"')
b1.outcome('SOLID', 'maintainability', 'testability')
b1.context('IoT', '"internet of things"', '"Cyber-Physical Systems"')


query1 = b1.build()
print("Query 1 (core):")
print(b1.build_core())
print("\nQuery 1 (completa):")
print(query1)

Query 1 (core):
(MCU OR microcontroller OR "embedded software" OR "embedded system") AND (architecture OR "design pattern") AND (SOLID OR maintainability OR testability) AND (IoT OR "internet of things" OR "Cyber-Physical Systems")

Query 1 (completa):
TITLE-ABS-KEY((MCU OR microcontroller OR "embedded software" OR "embedded system") AND (architecture OR "design pattern") AND (SOLID OR maintainability OR testability) AND (IoT OR "internet of things" OR "Cyber-Physical Systems")) AND PUBYEAR > 2020 AND PUBYEAR < 2026 AND LANGUAGE(english) AND DOCTYPE(ar OR cp)


In [35]:
# b2 = QueryBuilder(backend=scopus_qb)
# b2.population('MCU', 'microcontroller', '"embedded software"', '"embedded system"')
# b2.intervention('architecture', '"design pattern"')
# b2.outcome('SOLID', 'extensibility', 'extensible', 'maintainability', 'maintainable', 'testability', 'testable')
# b2.context('IoT', '"internet of things"', '"Cyber-Physical Systems"')

b2 = QueryBuilder(backend=scopus_qb)
b2.population('MCU', 'microcontroller', '"embedded software"', '"embedded system"')
b2.intervention('architecture', '"design pattern"')
b2.outcome('SOLID', 'extensibility', 'maintainability', 'testability')
b2.context('IoT', '"internet of things"', '"Cyber-Physical Systems"')


query2 = b2.build()
print("Query 2 (core):")
print(b2.build_core())
print("\nQuery 2 (completa):")
print(query2)

Query 2 (core):
(MCU OR microcontroller OR "embedded software" OR "embedded system") AND (architecture OR "design pattern") AND (SOLID OR extensibility OR maintainability OR testability) AND (IoT OR "internet of things" OR "Cyber-Physical Systems")

Query 2 (completa):
TITLE-ABS-KEY((MCU OR microcontroller OR "embedded software" OR "embedded system") AND (architecture OR "design pattern") AND (SOLID OR extensibility OR maintainability OR testability) AND (IoT OR "internet of things" OR "Cyber-Physical Systems")) AND PUBYEAR > 2020 AND PUBYEAR < 2026 AND LANGUAGE(english) AND DOCTYPE(ar OR cp)


## 4. Executar as buscas

In [36]:
client = ScopusClient()  # lê SCOPUS_API_KEY do .env ou variável de ambiente

print("Buscando resultados para Query 1...")
result1 = client.search(query1, max_results=500)
print(f"  Recuperados: {len(result1)} / Total disponível: {result1.total_count}")

print("Buscando resultados para Query 2...")
result2 = client.search(query2, max_results=500)
print(f"  Recuperados: {len(result2)} / Total disponível: {result2.total_count}")

Buscando resultados para Query 1...
  Recuperados: 54 / Total disponível: 54
Buscando resultados para Query 2...
  Recuperados: 61 / Total disponível: 61


## 5. Comparar os resultados

In [37]:
diff = compare(result1, result2)
print(diff.summary())

Total in query 1 : 54
Total in query 2 : 61
Common           : 54
Only in query 1  : 0
Only in query 2  : 7


## 6. Visualizar os resultados

In [38]:
diff.show()

### Documentos em comum (54)

,year,title,authors,source
0,2024,Intelligent Turning Cyber-Physical Systems Modeling using SysML,Dey P.R.,IECON Proceedings Industrial Electronics Conference
1,2024,An Intelligent Solid Waste Classification and Monitoring Alert System using Deep Learning,Selvi S.,2024 International Conference on Integration of Emerging Technologies for the Digital World Icietdw 2024
2,2025,Design and Performance Comparison of Current References for a 32-bit Microcontroller Using 22nm Technology,Barcelos E.A.,2025 IEEE 16th Latin American Symposium on Circuits and Systems Lascas 2025 Proceedings
3,2025,Real-Time Wireless Emergency Communication for Hazardous Environments,Ganesh C.S.S.,Proceedings of 7th International Conference on Inventive Material Science and Applications Icima 2025
4,2025,Model of an Open-Source MicroPython Library for GSM NB-IoT,Lupandin A.,Sensors
5,2025,EGaIn-Based Liquid Antennas: Beam Steering and Frequency Reconfiguration Using Microfluidic Control,Spyros L.,Proceedings of IEEE Computer Society Annual Symposium on VLSI Isvlsi
6,2025,An Intelligent Gamified Reviewer System for Enhancing Student Engagement and Performance in Cyber-Physical Learning Environments,Saballa-Marin M.J.M.,International Conference on Communication Computing Networking and Control in Cyber Physical Systems Ccncps 2025
7,2025,Smart Waste Management System an IoT Driven Framework,Chandra L.,2025 6th International Conference on Data Intelligence and Cognitive Informatics Icdici 2025
8,2025,"A CPS-Based Architecture for Mobile Robotics: Design, Integration, and Localisation Experiments",Líšková D.,Sensors
9,2025,Lightweight Embedded IoT Gateway for Smart Homes Based on an ESP32 Microcontroller,Serepas F.,Computers


### Apenas na query 1 (0)

,year,title,authors,source


### Apenas na query 2 (7)

,year,title,authors,source
0,2025,Deployment of a Machine Learning-SDN Pipeline for IoT Threat Detection in Smart City Environments,Antonio A.W.,IEEE Workshop on Local and Metropolitan Area Networks
1,2025,Benchmarking WebAssembly for Embedded Systems,Moron K.,ACM Transactions on Architecture and Code Optimization
2,2021,Optimizing Extensibility of CAN FD for Automotive Cyber-Physical Systems,Xie Y.,IEEE Transactions on Intelligent Transportation Systems
3,2021,Knowledge graphs as enhancers of intelligent digital twins,Sahlab N.,Proceedings 2021 4th IEEE International Conference on Industrial Cyber Physical Systems Icps 2021
4,2022,Formally verified architectural patterns of hybrid systems using proof and refinement with Event-B,Dupont G.,Science of Computer Programming
5,2023,Digital twin-based decision making paradigm of raise boring method,Hu F.,Journal of Intelligent Manufacturing
6,2023,ECS-Grid: Data-Oriented Real-Time Simulation Platform for Cyber-Physical Power Systems,Cheng T.,IEEE Transactions on Industrial Informatics
